# CNN Español Colombia — Scraper + LLM Enrichment + Database

Pipeline completo:
1. **Scraping** — extrae artículos de CNN Colombia (título, fecha, autor)
2. **Enriquecimiento LLM** — Claude extrae `topic` y `keywords` por artículo
3. **Base de datos** — guarda en SQLite + exporta CSV/Parquet

> **Prerequisito**: copia `.env.example` → `.env` y agrega tu `ANTHROPIC_API_KEY`.

## 0. Entorno — cargar `.env` y validar claves

In [ ]:
# DEBE IR PRIMERO — carga las variables del .env antes de cualquier import de cop_fx
import os
from dotenv import load_dotenv

load_dotenv()  # lee .env desde la raíz del proyecto (jupyter.notebookFileRoot = workspaceFolder)

api_key = os.getenv("ANTHROPIC_API_KEY", "")
if not api_key or api_key.startswith("sk-ant-..."):
    raise EnvironmentError(
        "ANTHROPIC_API_KEY no encontrada.\n"
        "Edita el archivo .env en la raíz del proyecto y agrega tu clave real."
    )
print(f"✓ ANTHROPIC_API_KEY cargada ({api_key[:12]}...)")

In [ ]:
import json
import sqlite3
from datetime import datetime, timezone
from pathlib import Path

import anthropic
import pandas as pd

from cop_fx.data.cnn_fetcher import CNNArticle, CNNColombiaFetcher

DB_PATH = Path("../data/cnn_articles.db")
DB_PATH.parent.mkdir(exist_ok=True)
print(f"Base de datos: {DB_PATH.resolve()}")

## 1. Scraping CNN Colombia

In [ ]:
fetcher = CNNColombiaFetcher(max_articles=30)

# enrich_authors=True → hace 1 request por artículo para obtener el autor real
articles: list[CNNArticle] = fetcher.fetch(enrich_authors=True)

print(f"\nArtículos obtenidos : {len(articles)}")
print(f"Con autor           : {sum(1 for a in articles if a.author)}")

In [ ]:
df_raw = pd.DataFrame([
    {
        "fecha"  : a.published_at.strftime("%Y-%m-%d"),
        "título" : a.title,
        "autor"  : a.author or "(sin autor)",
        "url"    : a.url,
    }
    for a in articles
])
df_raw

## 2. Enriquecimiento con Claude — `topic` y `keywords`

Usa **Haiku** (rápido y barato) para clasificar en lote. 
Tópicos posibles: `política`, `economía`, `deportes`, `internacional`, `sociedad`, `seguridad`, `tecnología`, `medio_ambiente`, `cultura`, `salud`.

In [ ]:
def enrich_with_llm(
    articles: list[CNNArticle],
    api_key: str,
    batch_size: int = 10,
) -> list[dict]:
    """Extrae topic y keywords para cada artículo usando Claude Haiku.
    
    Procesa en lotes para minimizar llamadas a la API.
    Devuelve lista ordenada igual que la entrada.
    """
    client = anthropic.Anthropic(api_key=api_key)
    all_results: list[dict] = []

    for start in range(0, len(articles), batch_size):
        batch = articles[start : start + batch_size]
        numbered = "\n".join(
            f"{i+1}. TÍTULO: {a.title}\n   RESUMEN: {(a.summary or '').strip()[:200] or '(sin resumen)'}"
            for i, a in enumerate(batch)
        )
        print(f"  Procesando lote {start // batch_size + 1} ({len(batch)} artículos)...")

        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1500,
            system=(
                "Eres un clasificador de noticias colombianas. "
                "Responde ÚNICAMENTE con un array JSON válido, sin texto adicional."
            ),
            messages=[{
                "role": "user",
                "content": (
                    "Clasifica cada noticia. Para cada una extrae:\n"
                    "- topic: UNA categoría de esta lista: política, economía, deportes, "
                    "internacional, sociedad, seguridad, tecnología, medio_ambiente, cultura, salud\n"
                    "- keywords: lista de 3 a 5 términos clave en español\n\n"
                    "Formato requerido (array JSON):\n"
                    '[{"id": 1, "topic": "economía", "keywords": ["dólar", "tasa", "COP"]}]\n\n'
                    f"Noticias:\n{numbered}"
                ),
            }],
        )

        raw = response.content[0].text.strip()
        # Extraer el JSON aunque Claude añada backticks
        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]
        batch_results: list[dict] = json.loads(raw)
        all_results.extend(batch_results)

    return all_results

In [ ]:
print("Enriqueciendo artículos con Claude Haiku...")
enrichment = enrich_with_llm(articles, api_key=api_key, batch_size=10)

# Unir resultados con artículos originales
enrichment_map = {r["id"]: r for r in enrichment}

rows = []
for i, article in enumerate(articles, start=1):
    meta = enrichment_map.get(i, {"topic": "sin_clasificar", "keywords": []})
    rows.append({
        "title"       : article.title,
        "author"      : article.author,
        "published_at": article.published_at.isoformat(),
        "summary"     : article.summary,
        "url"         : article.url,
        "source"      : article.source,
        "topic"       : meta.get("topic", "sin_clasificar"),
        "keywords"    : json.dumps(meta.get("keywords", []), ensure_ascii=False),
        "fetched_at"  : datetime.now(tz=timezone.utc).isoformat(),
    })

df = pd.DataFrame(rows)
print(f"\n✓ {len(df)} artículos enriquecidos")
print(df[["título" if "título" in df.columns else "title", "topic", "keywords"]]
      .rename(columns={"title": "título"}))

In [ ]:
# Distribución de tópicos
df["topic"].value_counts().rename("artículos").to_frame()

## 3. Base de datos SQLite

In [ ]:
def save_to_sqlite(df: pd.DataFrame, db_path: Path) -> int:
    """Inserta artículos nuevos (por URL) y devuelve cuántos se insertaron."""
    conn = sqlite3.connect(db_path)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS articles (
            id           INTEGER PRIMARY KEY AUTOINCREMENT,
            title        TEXT    NOT NULL,
            author       TEXT,
            published_at TEXT,
            summary      TEXT,
            url          TEXT    UNIQUE NOT NULL,
            source       TEXT,
            topic        TEXT,
            keywords     TEXT,
            fetched_at   TEXT
        )
    """)
    conn.commit()

    inserted = 0
    for _, row in df.iterrows():
        try:
            conn.execute(
                """
                INSERT INTO articles
                    (title, author, published_at, summary, url, source, topic, keywords, fetched_at)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
                """,
                (
                    row["title"], row["author"], row["published_at"],
                    row["summary"], row["url"], row["source"],
                    row["topic"], row["keywords"], row["fetched_at"],
                ),
            )
            inserted += 1
        except sqlite3.IntegrityError:
            pass  # URL duplicada — ya existe en la DB

    conn.commit()
    conn.close()
    return inserted


nuevos = save_to_sqlite(df, DB_PATH)
print(f"✓ {nuevos} artículos nuevos insertados en {DB_PATH}")
print(f"  ({len(df) - nuevos} duplicados ignorados)")

## 4. Consultar la base de datos

In [ ]:
conn = sqlite3.connect(DB_PATH)

# Todos los artículos
df_db = pd.read_sql("SELECT * FROM articles ORDER BY published_at DESC", conn)
conn.close()

print(f"Total en la DB: {len(df_db)} artículos")
df_db[["title", "author", "published_at", "topic", "keywords"]]

In [ ]:
# Filtrar por tópico
topic_filter = "economía"   # cambia según necesites

conn = sqlite3.connect(DB_PATH)
df_topic = pd.read_sql(
    "SELECT title, author, published_at, keywords FROM articles WHERE topic = ?",
    conn,
    params=(topic_filter,),
)
conn.close()

print(f"Artículos de '{topic_filter}': {len(df_topic)}")
df_topic

## 5. Exportar

In [ ]:
export_path = Path("../data")

# CSV
csv_file = export_path / "cnn_articles.csv"
df_db.to_csv(csv_file, index=False)
print(f"✓ CSV  → {csv_file}")

# Parquet (más eficiente para subir a cloud / BigQuery)
parquet_file = export_path / "cnn_articles.parquet"
df_db.to_parquet(parquet_file, index=False)
print(f"✓ Parquet → {parquet_file}")